# Install Required Libraries

In [22]:
%pip install google-genai tavily-python python-dotenv pydantic requests

Note: you may need to restart the kernel to use updated packages.


# Imports & Environment Setup

In [23]:
import os
import requests
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel
from typing import List
from datetime import datetime

# Load API Keys

In [24]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini Loaded:", GEMINI_API_KEY is not None)
print("Tavily Loaded:", TAVILY_API_KEY is not None)

Gemini Loaded: True
Tavily Loaded: True


# Tavily Search Function

In [25]:
def tavily_search(query: str, max_results: int = 3) -> str:
    
    url = "https://api.tavily.com/search"
    
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": max_results
    }
    
    response = requests.post(url, json=payload)
    data = response.json()
    
    if "results" not in data:
        return "No results found."
    
    results_text = ""
    
    for result in data["results"]:
        results_text += f"Title: {result['title']}\n"
        results_text += f"Content: {result['content']}\n\n"
    
    return results_text

# GUARDRAIL IMPLEMENTATION

In [26]:
class PoliticalGuardrailOutput(BaseModel):
    is_political: bool
    reasoning: str

In [ ]:
def political_guardrail(user_query: str) -> PoliticalGuardrailOutput:
    
    prompt = f"""
    Check if the following user query involves politics, politicians,
    elections, government officials, or geopolitical topics.
    
    Query:
    {user_query}
    
    Respond in this format:
    
    is_political: true/false
    reasoning: explanation
    """
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    text = response.text.lower()
    
    is_political = "true" in text
    
    return PoliticalGuardrailOutput(
        is_political=is_political,
        reasoning=response.text
    )

# Planner Agent (With Guardrail)

In [28]:
class SearchPlanItem(BaseModel):
    reason: str
    query: str

In [ ]:
def planner_agent(user_query: str) -> List[SearchPlanItem]:
    
    # Guardrail Check
    guardrail_result = political_guardrail(user_query)
    
    if guardrail_result.is_political:
        raise Exception(f"Guardrail Triggered: {guardrail_result.reasoning}")
    
    today = datetime.today().strftime("%Y-%m-%d")
    
    prompt = f"""
    Current Date: {today}
    
    You are a research planner agent.
    
    Break down the following query into 3 web search queries.
    
    For each include:
    - reason
    - search query
    
    User Query:
    {user_query}
    """
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    # Simplified parsing
    items = []
    sections = response.text.split("\n\n")
    
    for section in sections[:3]:
        lines = section.split("\n")
        if len(lines) >= 2:
            items.append(SearchPlanItem(
                reason=lines[0],
                query=lines[1]
            ))
    
    return items

# Define Search Agent

In [ ]:
def search_agent(query: str) -> str:
    
    search_results = tavily_search(query)
    
    prompt = f"""
    Summarize the following search results in less than 200 words.
    
    {search_results}
    """
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    return response.text

# Define Fundamentals Agent

In [ ]:
def fundamentals_agent(company: str) -> str:
    
    search_results = tavily_search(f"{company} financial statements revenue debt growth")
    
    prompt = f"""
    You are a financial analyst.
    
    Analyze the company's fundamentals from the notes below.
    
    {search_results}
    
    Provide summary in less than 200 words.
    """
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    return response.text

# Writer Agent

In [32]:
class FinalReport(BaseModel):
    executive_summary: str
    markdown_report: str
    follow_up_questions: List[str]

In [ ]:
def writer_agent(user_query: str, search_plan: List[SearchPlanItem]) -> FinalReport:
    
    search_summaries = ""
    
    # Mandatory Search Tool
    for item in search_plan:
        search_summaries += search_agent(item.query) + "\n\n"
    
    # Optional fundamentals call
    fundamentals_summary = ""
    if "fundamental" in user_query.lower():
        fundamentals_summary = fundamentals_agent(user_query)
    
    prompt = f"""
    You are an expert investment research writer.
    
    User Query:
    {user_query}
    
    Search Summaries:
    {search_summaries}
    
    Fundamentals (optional):
    {fundamentals_summary}
    
    Generate:
    1. Executive summary (2-3 sentences)
    2. Detailed markdown report (min 600 words)
    3. 3-5 follow-up research questions
    """
    
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    full_text = response.text
    
    return FinalReport(
        executive_summary=full_text,
        markdown_report=full_text,
        follow_up_questions=[]
    )

# Handoff Implementation

In [34]:
def planner_to_writer_handoff(user_query: str):
    
    print("Running Planner Agent...")
    
    search_plan = planner_agent(user_query)
    
    print("Handoff to Writer Agent...")
    
    final_report = writer_agent(user_query, search_plan)
    
    return final_report

In [35]:
from IPython.display import display, Markdown

In [36]:
def print_markdown(text):
    display(Markdown(text))

In [37]:
final_report = planner_to_writer_handoff(
    "Do a deep dive on latest developments in Apple stock. Include fundamental analysis."
)
print_markdown("### Executive Summary\n" + final_report.executive_summary)
print_markdown("### Detailed Report\n" + final_report.markdown_report)

Running Planner Agent...


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}